# Mejores Parámetros

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "reamember").exists():
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from omegaconf import OmegaConf
import json
import pandas as pd

from reamember.config import setGlobalSeed, setDeviceConfig
from reamember.datasets.text import TextDatasetWrapper
from reamember.eam.associative import NumpyAssociativeMemory as AssociativeMemory
from reamember.eam.mops import memorize
from reamember.pipes.text import (
    apply_text_noise,
    create_sonar_model,
    get_memory_batch_size,
    normalize_noise_level,
    text_reconstruction_metrics,
    get_scalar_config_value,
    get_experiment_path,
    load_embeddings_dataset,
    Quant,
 )

EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments" 

## Experimento 1

In [3]:
summary_path = EXPERIMENTS_ROOT / "npvinHnivqn-EnglishDictionary/definition_1024/recognition_noise_0.003/recognition_confusion_summary_True.json"

with open(summary_path, "r") as f:
    summary_data = json.load(f)

edf1 = pd.json_normalize(summary_data, sep="_")
edf1 = edf1.sort_values(["sigma", "xi", "kappa"]).reset_index(drop=True)

In [4]:
edf1

,dataset,latent_dim,memory_domain,sigma,iota,kappa,xi,seen_source,unseen_source,test_source,...,labels_columns,counts_seen_total,counts_unseen_total,counts_test_total,rates_seen_recognized_rate,rates_seen_unrecognized_rate,rates_unseen_recognized_rate,rates_unseen_unrecognized_rate,rates_test_recognized_rate,rates_test_unrecognized_rate
0,npvinHnivqn/EnglishDictionary,1024,4096,0.01,0.0,0.0,0.0,clean_train,noised_train,noised_test,...,"[recognized, unrecognized]",100440,100440,11161,1.000000,0.000000,0.323775,0.676225,0.320939,0.679061
1,npvinHnivqn/EnglishDictionary,1024,4096,0.01,0.1,0.0,0.0,clean_train,noised_train,noised_test,...,"[recognized, unrecognized]",100440,100440,11161,0.221207,0.778793,0.064486,0.935514,0.067377,0.932623
2,npvinHnivqn/EnglishDictionary,1024,4096,0.01,0.5,0.0,0.0,clean_train,noised_train,noised_test,...,"[recognized, unrecognized]",100440,100440,11161,0.001444,0.998556,0.000139,0.999861,0.000269,0.999731
3,npvinHnivqn/EnglishDictionary,1024,4096,0.01,1.0,0.0,0.0,clean_train,noised_train,noised_test,...,"[recognized, unrecognized]",100440,100440,11161,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
4,npvinHnivqn/EnglishDictionary,1024,8192,0.01,0.0,0.0,0.0,clean_train,noised_train,noised_test,...,"[recognized, unrecognized]",100440,100440,11161,1.000000,0.000000,0.155167,0.844833,0.159126,0.840874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,npvinHnivqn/EnglishDictionary,1024,8192,0.10,1.0,1.0,50.0,clean_train,noised_train,noised_test,...,"[recognized, unrecognized]",100440,100440,11161,0.059000,0.941000,0.008732,0.991268,0.009229,0.990771
320,npvinHnivqn/EnglishDictionary,1024,16384,0.10,0.0,1.0,50.0,clean_train,noised_train,noised_test,...,"[recognized, unrecognized]",100440,100440,11161,1.000000,0.000000,0.918917,0.081083,0.921064,0.078936
321,npvinHnivqn/EnglishDictionary,1024,16384,0.10,0.1,1.0,50.0,clean_train,noised_train,noised_test,...,"[recognized, unrecognized]",100440,100440,11161,0.931163,0.068837,0.736708,0.263292,0.740167,0.259833
322,npvinHnivqn/EnglishDictionary,1024,16384,0.10,0.5,1.0,50.0,clean_train,noised_train,noised_test,...,"[recognized, unrecognized]",100440,100440,11161,0.382248,0.617752,0.122889,0.877111,0.124451,0.875549


In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

matrix_rows = []
for record in summary_data:
    for row_index, source in enumerate(record["labels"]["rows"]):
        for col_index, outcome in enumerate(record["labels"]["columns"]):
            matrix_rows.append(
                {
                    "memory_domain": record["memory_domain"],
                    "sigma": record["sigma"],
                    "xi": record["xi"],
                    "iota": record["iota"],
                    "kappa": record["kappa"],
                    "source": source,
                    "outcome": outcome,
                    "count": record["matrix"][row_index][col_index],
                    "rate": record["rates"][f"{source}_{outcome}_rate"],
                }
            )

matrix_df = pd.DataFrame(matrix_rows)
source_order = ["seen", "unseen"]
outcome_order = ["recognized", "unrecognized"]
matrix_df = matrix_df[matrix_df["source"].isin(source_order)].copy()

memory_domain_values = sorted(matrix_df["memory_domain"].unique().tolist())
sigma_values = sorted(matrix_df["sigma"].unique().tolist())
xi_values = sorted(matrix_df["xi"].unique().tolist())
iota_values = sorted(matrix_df["iota"].unique().tolist())
kappa_values = sorted(matrix_df["kappa"].unique().tolist())

selected_xi = xi_values[0]
selected_iota = iota_values[0]
selected_kappa = kappa_values[0]
value_field = "rate"

subset = matrix_df[
    (matrix_df["xi"] == selected_xi)
    & (matrix_df["iota"] == selected_iota)
    & (matrix_df["kappa"] == selected_kappa)
].copy()

if subset.empty:
    raise ValueError(
        f"No data for xi={selected_xi:g}, iota={selected_iota:g}, kappa={selected_kappa:g}."
    )

subset["source"] = pd.Categorical(subset["source"], categories=source_order, ordered=True)
subset["outcome"] = pd.Categorical(subset["outcome"], categories=outcome_order, ordered=True)
subset = subset.sort_values(["memory_domain", "sigma", "source", "outcome"])

if value_field == "rate":
    zmin, zmax = 0.0, 1.0
    colorscale = "YlOrRd"
    colorbar_title = "rate"

    def formatter(value):
        return f"{value:.3f}"
else:
    zmin, zmax = 0, subset["count"].max()
    colorscale = "Blues"
    colorbar_title = "count"

    def formatter(value):
        return f"{int(value)}"

fig = make_subplots(
    rows=len(memory_domain_values),
    cols=len(sigma_values),
    row_titles=[f"msize={memory_domain}" for memory_domain in memory_domain_values],
    column_titles=[f"sigma={sigma:g}" for sigma in sigma_values],
    horizontal_spacing=0.06,
    vertical_spacing=0.10,
)

for row_number, memory_domain in enumerate(memory_domain_values, start=1):
    for col_number, sigma in enumerate(sigma_values, start=1):
        panel = subset[
            (subset["memory_domain"] == memory_domain)
            & (subset["sigma"] == sigma)
        ]

        if panel.empty:
            fig.add_annotation(
                row=row_number,
                col=col_number,
                text="sin datos",
                showarrow=False,
                font={"size": 11, "color": "gray"},
            )
            continue

        matrix_view = (
            panel.pivot(index="source", columns="outcome", values=value_field)
            .reindex(index=source_order, columns=outcome_order)
        )
        text_values = matrix_view.map(formatter)

        fig.add_trace(
            go.Heatmap(
                z=matrix_view.to_numpy(),
                x=outcome_order,
                y=source_order,
                zmin=zmin,
                zmax=zmax,
                colorscale=colorscale,
                showscale=False,
                text=text_values.to_numpy(),
                texttemplate="%{text}",
                hovertemplate=(
                    "source=%{y}<br>prediction=%{x}<br>"
                    + f"{colorbar_title}=%{{z}}<extra></extra>"
                ),
            ),
            row=row_number,
            col=col_number,
        )

fig.update_xaxes(title_text="")
fig.update_yaxes(title_text="")
fig.update_annotations(font={"size": 11})
fig.update_layout(
    title=(
        "Experiment 1: Recall Test"
        + f" | xi={selected_xi:g}, iota={selected_iota:g}, kappa={selected_kappa:g}"
    ),
    height=340 * len(memory_domain_values),
    width=360 * len(sigma_values),
    margin={"l": 120, "r": 40, "t": 120, "b": 80},
    annotations=[
        *fig.layout.annotations,
        dict(
            text="prediction",
            x=0.5,
            y=-0.08,
            xref="paper",
            yref="paper",
            showarrow=False,
            font={"size": 12},
        ),
        dict(
            text="source split",
            x=-0.10,
            y=0.5,
            xref="paper",
            yref="paper",
            textangle=-90,
            showarrow=False,
            font={"size": 12},
        ),
    ],
)
fig.show()

In [6]:
# Guardar gráfica por sigma
fig.write_image("confusion_by_sigma.svg")

In [7]:
selected_sigma = sigma_values[0]
selected_xi = xi_values[0]
selected_iota = iota_values[0]
value_field = "rate"

subset = matrix_df[
    (matrix_df["sigma"] == selected_sigma)
    & (matrix_df["xi"] == selected_xi)
    & (matrix_df["iota"] == selected_iota)
].copy()

if subset.empty:
    raise ValueError(
        f"No data for sigma={selected_sigma:g}, xi={selected_xi:g}, iota={selected_iota:g}."
    )

subset["source"] = pd.Categorical(subset["source"], categories=source_order, ordered=True)
subset["outcome"] = pd.Categorical(subset["outcome"], categories=outcome_order, ordered=True)
subset = subset.sort_values(["memory_domain", "kappa", "source", "outcome"])

if value_field == "rate":
    zmin, zmax = 0.0, 1.0
    colorscale = "YlOrRd"
    colorbar_title = "rate"

    def formatter(value):
        return f"{value:.3f}"
else:
    zmin, zmax = 0, subset["count"].max()
    colorscale = "Blues"
    colorbar_title = "count"

    def formatter(value):
        return f"{int(value)}"

fig = make_subplots(
    rows=len(memory_domain_values),
    cols=len(kappa_values),
    row_titles=[f"msize={memory_domain}" for memory_domain in memory_domain_values],
    column_titles=[f"kappa={kappa:g}" for kappa in kappa_values],
    horizontal_spacing=0.06,
    vertical_spacing=0.10,
)

for row_number, memory_domain in enumerate(memory_domain_values, start=1):
    for col_number, kappa in enumerate(kappa_values, start=1):
        panel = subset[
            (subset["memory_domain"] == memory_domain)
            & (subset["kappa"] == kappa)
        ]

        if panel.empty:
            fig.add_annotation(
                row=row_number,
                col=col_number,
                text="sin datos",
                showarrow=False,
                font={"size": 11, "color": "gray"},
            )
            continue

        matrix_view = (
            panel.pivot(index="source", columns="outcome", values=value_field)
            .reindex(index=source_order, columns=outcome_order)
        )
        text_values = matrix_view.map(formatter)

        fig.add_trace(
            go.Heatmap(
                z=matrix_view.to_numpy(),
                x=outcome_order,
                y=source_order,
                zmin=zmin,
                zmax=zmax,
                colorscale=colorscale,
                showscale=False,
                text=text_values.to_numpy(),
                texttemplate="%{text}",
                hovertemplate=(
                    "source=%{y}<br>prediction=%{x}<br>"
                    + f"{colorbar_title}=%{{z}}<extra></extra>"
                ),
            ),
            row=row_number,
            col=col_number,
        )

fig.update_xaxes(title_text="")
fig.update_yaxes(title_text="")
fig.update_annotations(font={"size": 11})
fig.update_layout(
    title=(
        "Experiment 1: Recall Test"
        + f" | sigma={selected_sigma:g}, xi={selected_xi:g}, iota={selected_iota:g}"
    ),
    height=340 * len(memory_domain_values),
    width=360 * len(kappa_values),
    margin={"l": 120, "r": 40, "t": 120, "b": 80},
    annotations=[
        *fig.layout.annotations,
        dict(
            text="prediction",
            x=0.5,
            y=-0.08,
            xref="paper",
            yref="paper",
            showarrow=False,
            font={"size": 12},
        ),
        dict(
            text="source split",
            x=-0.10,
            y=0.5,
            xref="paper",
            yref="paper",
            textangle=-90,
            showarrow=False,
            font={"size": 12},
        ),
    ],
)
fig.show()

In [8]:
# Guardar gráfica por kappa
fig.write_image("confusion_by_kappa.svg")

In [9]:
selected_sigma = sigma_values[0]
selected_iota = iota_values[0]
selected_kappa = kappa_values[0]
value_field = "rate"

subset = matrix_df[
    (matrix_df["sigma"] == selected_sigma)
    & (matrix_df["iota"] == selected_iota)
    & (matrix_df["kappa"] == selected_kappa)
].copy()

if subset.empty:
    raise ValueError(
        f"No data for sigma={selected_sigma:g}, iota={selected_iota:g}, kappa={selected_kappa:g}."
    )

subset["source"] = pd.Categorical(subset["source"], categories=source_order, ordered=True)
subset["outcome"] = pd.Categorical(subset["outcome"], categories=outcome_order, ordered=True)
subset = subset.sort_values(["memory_domain", "xi", "source", "outcome"])

if value_field == "rate":
    zmin, zmax = 0.0, 1.0
    colorscale = "YlOrRd"
    colorbar_title = "rate"

    def formatter(value):
        return f"{value:.3f}"
else:
    zmin, zmax = 0, subset["count"].max()
    colorscale = "Blues"
    colorbar_title = "count"

    def formatter(value):
        return f"{int(value)}"

fig = make_subplots(
    rows=len(memory_domain_values),
    cols=len(xi_values),
    row_titles=[f"msize={memory_domain}" for memory_domain in memory_domain_values],
    column_titles=[f"xi={xi:g}" for xi in xi_values],
    horizontal_spacing=0.06,
    vertical_spacing=0.10,
)

for row_number, memory_domain in enumerate(memory_domain_values, start=1):
    for col_number, xi in enumerate(xi_values, start=1):
        panel = subset[
            (subset["memory_domain"] == memory_domain)
            & (subset["xi"] == xi)
        ]

        if panel.empty:
            fig.add_annotation(
                row=row_number,
                col=col_number,
                text="sin datos",
                showarrow=False,
                font={"size": 11, "color": "gray"},
            )
            continue

        matrix_view = (
            panel.pivot(index="source", columns="outcome", values=value_field)
            .reindex(index=source_order, columns=outcome_order)
        )
        text_values = matrix_view.map(formatter)

        fig.add_trace(
            go.Heatmap(
                z=matrix_view.to_numpy(),
                x=outcome_order,
                y=source_order,
                zmin=zmin,
                zmax=zmax,
                colorscale=colorscale,
                showscale=False,
                text=text_values.to_numpy(),
                texttemplate="%{text}",
                hovertemplate=(
                    "source=%{y}<br>prediction=%{x}<br>"
                    + f"{colorbar_title}=%{{z}}<extra></extra>"
                ),
            ),
            row=row_number,
            col=col_number,
        )

fig.update_xaxes(title_text="")
fig.update_yaxes(title_text="")
fig.update_annotations(font={"size": 11})
fig.update_layout(
    title=(
        "Experiment 1: Recall Test"
        + f" | sigma={selected_sigma:g}, iota={selected_iota:g}, kappa={selected_kappa:g}"
    ),
    height=340 * len(memory_domain_values),
    width=360 * len(xi_values),
    margin={"l": 120, "r": 40, "t": 120, "b": 80},
    annotations=[
        *fig.layout.annotations,
        dict(
            text="prediction",
            x=0.5,
            y=-0.08,
            xref="paper",
            yref="paper",
            showarrow=False,
            font={"size": 12},
        ),
        dict(
            text="source split",
            x=-0.10,
            y=0.5,
            xref="paper",
            yref="paper",
            textangle=-90,
            showarrow=False,
            font={"size": 12},
        ),
    ],
)
fig.show()

In [10]:
# Guardar gráfica por xi
fig.write_image("confusion_by_xi.svg")

In [11]:
selected_sigma = sigma_values[0]
selected_xi = xi_values[0]
selected_kappa = kappa_values[0]
value_field = "rate"

subset = matrix_df[
    (matrix_df["sigma"] == selected_sigma)
    & (matrix_df["xi"] == selected_xi)
    & (matrix_df["kappa"] == selected_kappa)
].copy()

if subset.empty:
    raise ValueError(
        f"No data for sigma={selected_sigma:g}, xi={selected_xi:g}, kappa={selected_kappa:g}."
    )

subset["source"] = pd.Categorical(subset["source"], categories=source_order, ordered=True)
subset["outcome"] = pd.Categorical(subset["outcome"], categories=outcome_order, ordered=True)
subset = subset.sort_values(["memory_domain", "iota", "source", "outcome"])

if value_field == "rate":
    zmin, zmax = 0.0, 1.0
    colorscale = "YlOrRd"
    colorbar_title = "rate"

    def formatter(value):
        return f"{value:.3f}"
else:
    zmin, zmax = 0, subset["count"].max()
    colorscale = "Blues"
    colorbar_title = "count"

    def formatter(value):
        return f"{int(value)}"

fig = make_subplots(
    rows=len(memory_domain_values),
    cols=len(iota_values),
    row_titles=[f"msize={memory_domain}" for memory_domain in memory_domain_values],
    column_titles=[f"iota={iota:g}" for iota in iota_values],
    horizontal_spacing=0.06,
    vertical_spacing=0.10,
)

for row_number, memory_domain in enumerate(memory_domain_values, start=1):
    for col_number, iota in enumerate(iota_values, start=1):
        panel = subset[
            (subset["memory_domain"] == memory_domain)
            & (subset["iota"] == iota)
        ]

        if panel.empty:
            fig.add_annotation(
                row=row_number,
                col=col_number,
                text="sin datos",
                showarrow=False,
                font={"size": 11, "color": "gray"},
            )
            continue

        matrix_view = (
            panel.pivot(index="source", columns="outcome", values=value_field)
            .reindex(index=source_order, columns=outcome_order)
        )
        text_values = matrix_view.map(formatter)

        fig.add_trace(
            go.Heatmap(
                z=matrix_view.to_numpy(),
                x=outcome_order,
                y=source_order,
                zmin=zmin,
                zmax=zmax,
                colorscale=colorscale,
                showscale=False,
                text=text_values.to_numpy(),
                texttemplate="%{text}",
                hovertemplate=(
                    "source=%{y}<br>prediction=%{x}<br>"
                    + f"{colorbar_title}=%{{z}}<extra></extra>"
                ),
            ),
            row=row_number,
            col=col_number,
        )

fig.update_xaxes(title_text="")
fig.update_yaxes(title_text="")
fig.update_annotations(font={"size": 11})
fig.update_layout(
    title=(
        "Experiment 1: Recall Test"
        + f" | sigma={selected_sigma:g}, xi={selected_xi:g}, kappa={selected_kappa:g}"
    ),
    height=340 * len(memory_domain_values),
    width=360 * len(iota_values),
    margin={"l": 120, "r": 40, "t": 120, "b": 80},
    annotations=[
        *fig.layout.annotations,
        dict(
            text="prediction",
            x=0.5,
            y=-0.08,
            xref="paper",
            yref="paper",
            showarrow=False,
            font={"size": 12},
        ),
        dict(
            text="source split",
            x=-0.10,
            y=0.5,
            xref="paper",
            yref="paper",
            textangle=-90,
            showarrow=False,
            font={"size": 12},
        ),
    ],
)
fig.show()

In [12]:
# Guardar gráfica por iota
fig.write_image("confusion_by_iota.svg")

## Experimento 2

In [13]:

data = EXPERIMENTS_ROOT / "npvinHnivqn-EnglishDictionary/definition_1024/global_memories_results_noise_0.003.json"

with open(data, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

In [14]:
df

,latent,msize,filling_percent,sigma,xi,iota,kappa,recognized,unrecognized,mean_cosine,mean_euclidean,mean_l2,mean_edit_distance
0,1024,4096,1.0,0.01,0.0,0.0,0.0,0.339307,0.660693,0.950606,0.052753,0.052753,38.954846
1,1024,4096,1.0,0.01,0.0,0.0,0.5,0.339307,0.660693,0.950884,0.052560,0.052560,38.868762
2,1024,4096,1.0,0.01,0.0,0.0,1.0,0.339307,0.660693,0.950681,0.052687,0.052687,38.853974
3,1024,4096,1.0,0.01,0.0,0.1,0.0,0.067377,0.932623,0.959482,0.042574,0.042574,39.396277
4,1024,4096,1.0,0.01,0.0,0.1,0.5,0.067377,0.932623,0.959572,0.042289,0.042289,39.472074
...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,1024,16384,1.0,0.10,50.0,0.5,0.5,0.134397,0.865603,0.468364,0.191491,0.191491,59.103333
320,1024,16384,1.0,0.10,50.0,0.5,1.0,0.134397,0.865603,0.468047,0.191973,0.191973,59.194667
321,1024,16384,1.0,0.10,50.0,1.0,0.0,0.000986,0.999014,0.417195,0.195806,0.195806,59.636364
322,1024,16384,1.0,0.10,50.0,1.0,0.5,0.000986,0.999014,0.483014,0.188576,0.188576,61.090909


In [15]:
plot_df = df[["sigma", "mean_cosine", "mean_euclidean", "mean_edit_distance"]].copy()

In [16]:
plot_df = df[[
    "sigma",
    "xi",
    "iota",
    "kappa",
    "latent",
    "msize",
    "filling_percent",
    "recognized",
    "unrecognized",
    "mean_cosine",
    "mean_euclidean",
    "mean_edit_distance",
]].copy()
plot_df["msize_label"] = plot_df["msize"].astype(str)
plot_df["xi_label"] = plot_df["xi"].astype(str)
msize_order = [str(msize) for msize in sorted(plot_df["msize"].unique())]
xi_order = [str(xi) for xi in sorted(plot_df["xi"].unique())]
plot_df = plot_df.sort_values(["msize", "sigma", "xi"])

plot_df_long = plot_df.melt(
    id_vars=[
        "sigma",
        "xi",
        "xi_label",
        "iota",
        "kappa",
        "latent",
        "msize",
        "msize_label",
        "filling_percent",
        "recognized",
        "unrecognized",
    ],
    value_vars=["mean_cosine", "mean_euclidean", "mean_edit_distance"],
    var_name="metric",
    value_name="value",
)

fig = px.scatter(
    plot_df_long,
    x="sigma",
    y="value",
    facet_row="metric",
    facet_col="msize_label",
    color="xi_label",
    size="recognized",
    category_orders={
        "msize_label": msize_order,
        "xi_label": xi_order,
    },
    hover_data={
        "xi_label": False,
        "msize_label": False,
        "xi": True,
        "iota": True,
        "kappa": True,
        "latent": True,
        "msize": True,
        "filling_percent": True,
        "recognized": True,
        "unrecognized": True,
    },
    title="Sigma vs reconstruction metrics by msize",
    height=900,
)
fig.update_yaxes(matches=None)
fig.update_layout(
    xaxis_title="sigma",
    yaxis_title="value",
    legend_title_text="xi",
)
fig.show()

In [17]:
# Guardar gráfica por sigma como SVG y HTML

fig.write_image("metrics_by_msize_and_sigma.svg")

In [18]:
plot_df = df[[
    "sigma",
    "xi",
    "iota",
    "kappa",
    "latent",
    "msize",
    "filling_percent",
    "recognized",
    "unrecognized",
    "mean_cosine",
    "mean_euclidean",
    "mean_edit_distance",
]].copy()
plot_df["msize_label"] = plot_df["msize"].astype(str)
plot_df["sigma_label"] = plot_df["sigma"].astype(str)
msize_order = [str(msize) for msize in sorted(plot_df["msize"].unique())]
sigma_order = [str(sigma) for sigma in sorted(plot_df["sigma"].unique())]
plot_df = plot_df.sort_values(["msize", "xi", "sigma"])

plot_df_long = plot_df.melt(
    id_vars=[
        "sigma",
        "sigma_label",
        "xi",
        "iota",
        "kappa",
        "latent",
        "msize",
        "msize_label",
        "filling_percent",
        "recognized",
        "unrecognized",
    ],
    value_vars=["mean_cosine", "mean_euclidean", "mean_edit_distance"],
    var_name="metric",
    value_name="value",
)

fig = px.scatter(
    plot_df_long,
    x="xi",
    y="value",
    facet_row="metric",
    facet_col="msize_label",
    color="sigma_label",
    size="recognized",
    category_orders={
        "msize_label": msize_order,
        "sigma_label": sigma_order,
    },
    hover_data={
        "sigma_label": False,
        "msize_label": False,
        "sigma": True,
        "iota": True,
        "kappa": True,
        "latent": True,
        "msize": True,
        "filling_percent": True,
        "recognized": True,
        "unrecognized": True,
    },
    title="Xi vs reconstruction metrics by msize",
    height=900,
)
fig.update_yaxes(matches=None)
fig.update_layout(
    xaxis_title="xi",
    yaxis_title="value",
    legend_title_text="sigma",
)
fig.show()

In [19]:
# Guardar como SVG

fig.write_image("metrics_by_msize_and_xi.svg")

In [20]:
# Ahora filtremos solo as que tengan mas de 90% de recognized

filtered_plot_df = plot_df[plot_df["recognized"] / (plot_df["recognized"] + plot_df["unrecognized"]) >= 0.9].copy()

# Ordenemos por filling_percent dentro de cada msize

filtered_plot_df = filtered_plot_df.sort_values(["mean_cosine"], ascending=False)

print("Mejores resultados con >=90% recognized:")
print(filtered_plot_df[:10][[
    "sigma",
    "xi",
    "iota",
    "kappa",
    "latent",
    "msize",
    "filling_percent",
    "recognized",
    "unrecognized",
    "mean_cosine",
    "mean_euclidean",
    "mean_edit_distance",
]])

Mejores resultados con >=90% recognized:
     sigma    xi  iota  kappa  latent  msize  filling_percent  recognized  \
240   0.01  50.0   0.0    0.0    1024  16384              1.0    0.933071   
120   0.01  30.0   0.0    0.0    1024   8192              1.0    0.944718   
242   0.01  50.0   0.0    1.0    1024  16384              1.0    0.933071   
121   0.01  30.0   0.0    0.5    1024   8192              1.0    0.944718   
122   0.01  30.0   0.0    1.0    1024   8192              1.0    0.944718   
241   0.01  50.0   0.0    0.5    1024  16384              1.0    0.933071   
132   0.01  50.0   0.0    0.0    1024   8192              1.0    0.983693   
134   0.01  50.0   0.0    1.0    1024   8192              1.0    0.983693   
133   0.01  50.0   0.0    0.5    1024   8192              1.0    0.983693   
12    0.01  30.0   0.0    0.0    1024   4096              1.0    0.985216   

     unrecognized  mean_cosine  mean_euclidean  mean_edit_distance  
240      0.066929     0.951283        0.05

In [25]:
# Ahora grafiquemos los mejores resultados
fig = px.scatter(
    filtered_plot_df[:10],
    x="mean_cosine",
    y="mean_edit_distance",
    size="recognized",
    color="msize_label",
    hover_data={
        "sigma": True,
        "xi": True,
        "iota": True,
        "kappa": True,
        "latent": True,
        "msize": True,
        "filling_percent": True,
        "recognized": True,
        "unrecognized": True,
    },
    title="Best configurations with >=90% recognized",
)
fig.update_layout(
    xaxis_title="mean_cosine",
    yaxis_title="mean_edit_distance",
    legend_title_text="msize",
    width=1200,
    height=600,
)
fig.show()

In [26]:
fig.write_image("best_configs.svg", format="svg", scale=2)